# NBA RAG Embedding Fine-Tuning
## Thunder Applied AI Engineer Internship - Part 4

**Based on:** Wang et al. (2022) "Text Embeddings by Weakly-Supervised Contrastive Pre-training" [arXiv:2212.03533](https://arxiv.org/abs/2212.03533)

This notebook fine-tunes `intfloat/e5-base-v2` on NBA-specific query-context pairs to improve retrieval accuracy for the Thunder RAG pipeline.

## Step 1: Install Dependencies

In [ ]:
!pip install -q sentence-transformers

## Step 2: Upload Training Data

Upload `training_pairs.json` from your `part4/` folder.

In [ ]:
from google.colab import files
uploaded = files.upload()  # Upload training_pairs.json

## Step 3: Load and Verify Data

In [ ]:
import json

with open('training_pairs.json', 'r') as f:
    data = json.load(f)

# Split data
train_pairs = [p for p in data['training_pairs'] if p['split'] == 'train']
val_pairs = [p for p in data['training_pairs'] if p['split'] == 'validation']
test_pairs = [p for p in data['training_pairs'] if p['split'] == 'test']

print(f"✅ Training pairs: {len(train_pairs)}")
print(f"✅ Validation pairs: {len(val_pairs)}")
print(f"✅ Test pairs: {len(test_pairs)}")
print(f"\n📊 Total: {len(data['training_pairs'])} pairs")

# Preview examples
print(f"\n--- Sample Training Pair ---")
print(f"Query: {train_pairs[0]['query']}")
print(f"Context: {train_pairs[0]['positive'][:100]}...")

## Step 4: Fine-Tune E5 Model

### Paper Methodology (Wang et al. 2022, Section 4.1):

> "We use a shared encoder for all input texts and break the symmetry by adding two prefix identifiers 'query:' and 'passage:' to q and d respectively."

> "Here we choose to use the in-batch negatives, where the passages from other pairs in a batch serve as negative samples."

> "τ is set to 0.01 in our experiments by default."

In [ ]:
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader
from datetime import datetime

# Load base model
# Paper Table 10: "E5base: 12 layers, 768 hidden size, 110M parameters"
print("Loading intfloat/e5-base-v2...")
model = SentenceTransformer('intfloat/e5-base-v2')

# CRITICAL: E5 requires "query: " and "passage: " prefixes
# Paper Section 4.1: "adding two prefix identifiers 'query:' and 'passage:'"
train_examples = [
    InputExample(texts=[
        f"query: {p['query']}",      # PREFIX REQUIRED
        f"passage: {p['positive']}"   # PREFIX REQUIRED
    ])
    for p in train_pairs
]

# Paper Section 4.1: "in-batch negatives with a large batch-size"
# Table 5 shows batch size impact (32K > 8K > 1K)
# We use 8 due to small dataset
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=8)

# Paper Equation 1: InfoNCE contrastive loss
train_loss = losses.MultipleNegativesRankingLoss(model)

# Train
# Paper Table 11: 3 epochs for fine-tuning
print(f"\n🚀 Fine-tuning on {len(train_examples)} NBA examples...")
start = datetime.now()

model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=3,           # Paper Table 11
    warmup_steps=10,    # Scaled from paper's 400
    output_path='./e5-nba-finetuned',
    show_progress_bar=True
)

training_time = (datetime.now() - start).total_seconds()
print(f"\n✅ Training complete in {training_time:.1f} seconds")

## Step 5: Evaluate Against Baseline

Compare fine-tuned model to baseline E5 on held-out test queries.

In [ ]:
import numpy as np

def evaluate_model(model, test_pairs, corpus, corpus_indices):
    """
    Evaluate retrieval metrics: Recall@1, Recall@5, MRR
    
    Paper methodology:
    - Section 4.1: "query:" and "passage:" prefixes required
    - Section 4.1: Uses cosine similarity (Equation 2)
    """
    
    # Apply E5 prefixes (REQUIRED per paper Section 4.1)
    queries = [f"query: {p['query']}" for p in test_pairs]
    passages = [f"passage: {c}" for c in corpus]
    
    # Encode with normalization (enables cosine via dot product)
    query_emb = model.encode(queries, normalize_embeddings=True, show_progress_bar=False)
    corpus_emb = model.encode(passages, normalize_embeddings=True, show_progress_bar=False)
    
    recall_1, recall_5, mrr = 0, 0, 0
    
    for i, (q_emb, test_pair) in enumerate(zip(query_emb, test_pairs)):
        # Find correct answer index in corpus
        correct_idx = corpus_indices[test_pair['game_id']]
        
        # Compute similarities
        sims = np.dot(corpus_emb, q_emb)
        ranked = np.argsort(sims)[::-1]
        
        # Find rank of correct answer
        rank = np.where(ranked == correct_idx)[0][0] + 1
        
        if rank == 1: recall_1 += 1
        if rank <= 5: recall_5 += 1
        mrr += 1 / rank
    
    n = len(test_pairs)
    return {
        'Recall@1': recall_1 / n,
        'Recall@5': recall_5 / n,
        'MRR': mrr / n
    }

# Build corpus from ALL pairs (simulating full retrieval)
all_pairs = data['training_pairs']
corpus = [p['positive'] for p in all_pairs]
corpus_indices = {p['game_id']: i for i, p in enumerate(all_pairs)}

# Load both models
print("Loading baseline model...")
baseline = SentenceTransformer('intfloat/e5-base-v2')

print("Loading fine-tuned model...")
finetuned = SentenceTransformer('./e5-nba-finetuned')

# Evaluate
print("\n📊 Evaluating baseline (pre-trained E5)...")
baseline_metrics = evaluate_model(baseline, test_pairs, corpus, corpus_indices)

print("📊 Evaluating fine-tuned (E5-NBA)...")
finetuned_metrics = evaluate_model(finetuned, test_pairs, corpus, corpus_indices)

## Step 6: Display Results

In [ ]:
print("\n" + "="*70)
print("🏀 NBA RAG FINE-TUNING RESULTS")
print("="*70)
print(f"{'Metric':<12} {'E5-base-v2':<18} {'E5-NBA (tuned)':<18} {'Improvement':<12}")
print("-"*70)

for metric in ['Recall@1', 'Recall@5', 'MRR']:
    b = baseline_metrics[metric]
    f = finetuned_metrics[metric]
    imp = (f - b) / b * 100 if b > 0 else float('inf')
    print(f"{metric:<12} {b:<18.1%} {f:<18.1%} {imp:>+10.1f}%")

print("="*70)
print(f"\n📈 Paper context (Section 5.4): 'E5 models substantially outperform")
print(f"   existing ones with similar sizes when fine-tuned.'")

## Step 7: Save Results

In [ ]:
results = {
    "baseline": baseline_metrics,
    "finetuned": finetuned_metrics,
    "methodology": {
        "paper": "Wang et al. (2022) arXiv:2212.03533",
        "approach": "Contrastive fine-tuning with in-batch negatives (Section 4.1)",
        "loss": "InfoNCE (Equation 1)",
        "temperature": 0.01,
        "prefixes": "query: and passage: (Section 4.1)",
        "batch_size": 8,
        "epochs": 3,
        "training_time_seconds": training_time
    },
    "data": {
        "training_pairs": len(train_pairs),
        "validation_pairs": len(val_pairs),
        "test_pairs": len(test_pairs)
    }
}

with open('evaluation_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print("✅ Results saved to evaluation_results.json")

# Download results
files.download('evaluation_results.json')

## Step 8: (Optional) Test Individual Queries

In [ ]:
def test_query(query, model, corpus, top_k=3):
    """Test a single query against the corpus."""
    q_emb = model.encode(f"query: {query}", normalize_embeddings=True)
    c_emb = model.encode([f"passage: {c}" for c in corpus], normalize_embeddings=True)
    
    sims = np.dot(c_emb, q_emb)
    top_indices = np.argsort(sims)[::-1][:top_k]
    
    print(f"Query: {query}\n")
    for i, idx in enumerate(top_indices):
        print(f"#{i+1} (score: {sims[idx]:.3f})")
        print(f"   {corpus[idx][:150]}...\n")

# Test with evaluation questions
print("=" * 70)
print("FINE-TUNED MODEL RETRIEVAL TEST")
print("=" * 70 + "\n")

test_queries = [
    "How many points did the Warriors score against the Kings on October 27, 2023?",
    "Which team won the 2024 New Year's Eve game between Thunder and Timberwolves?",
    "How many points did Luka Dončić score against the Hawks on 1-26-24?",
]

for q in test_queries:
    test_query(q, finetuned, corpus)
    print("-" * 70 + "\n")

## Step 9: (Optional) Upload to Hugging Face Hub

In [ ]:
# Uncomment to upload your fine-tuned model to Hugging Face

# from huggingface_hub import login
# login()  # Enter your HF token

# HUB_NAME = "drbinna/e5-nba-finetuned"
# finetuned.save_to_hub(HUB_NAME)
# print(f"✅ Model uploaded to huggingface.co/{HUB_NAME}")